In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
p = pathlib.Path.cwd()
for q in (p, *p.parents):
    s = q / "src" / "ftbp"   # <- change "ftbp" if you rename the package
    if s.exists():
        sys.path.insert(0, str(s.parent))  # add .../src
        break
else:
    raise RuntimeError("src/ftbp not found")

In [ ]:
import numpy as np
import pandas as pd
import itertools
from math import comb
from scipy.optimize import brentq
from scipy.stats import norm, cauchy, uniform
from scipy.stats import median_abs_deviation
from ftbp.bootstrap import *
from ftbp.bootstrap import _compute_basics_bootstrap
from ftbp.io import *
import seaborn as sns
import matplotlib.pyplot as plt

## Real Data, Calcium vs Placebo

In [ ]:
# Run real data
calcium, placebo = read_data("calcium")
scaled_calcium = calcium / (1.4826 * median_abs_deviation(calcium))
scaled_placebo = placebo / (1.4826 * median_abs_deviation(placebo))
print(scaled_calcium, '\n', scaled_placebo)

### Bootstrap Plot for Fixed Std

In [ ]:
# Plot 1, show how the difference changes with different $m$
# Get bootstrap weights from standard exponential
np.random.seed(5353)
B = 10000
m_grid = list(range(0, 5))
alpha = 0.05 # does not affect eta here
loss_types = ['huber', 'logcosh', 'concordant']

dfs = []
for loss_type in loss_types:
    rows = []
    if loss_type == 'huber':
        delta = 1.345
    elif loss_type == 'logcosh':
        delta = 1.2047
    else:
        delta = 1.4811
    for m in m_grid:
        lowers = []
        uppers = []
        wx = np.ones(len(scaled_calcium), dtype=float); wy = np.ones(len(scaled_placebo), dtype=float)
        eta = eta_at_m_bootstrap(scaled_calcium, scaled_placebo, wx, wy, m=m, delta=delta, alpha=alpha, loss_type=loss_type, direction='positive')['eta']
        eta_bp = []
        for b in range(B):
            wxb = np.random.exponential(scale=1.0, size=len(scaled_calcium))
            wyb = np.random.exponential(scale=1.0, size=len(scaled_placebo))
            etab = eta_at_m_bootstrap(scaled_calcium, scaled_placebo, wxb, wyb, m=m, delta=delta, alpha=alpha, loss_type=loss_type, direction='positive')['eta']
            eta_bp.append(etab)
        eta_bp = np.array(eta_bp)
        rows.append({"eta": float(eta),
                    "eta_lower_95": float(np.percentile(eta_bp, 2.5)),
                    "eta_upper_95": float(np.percentile(eta_bp, 97.5)),
                    "eta_lower_80": float(np.percentile(eta_bp, 10)),
                    "eta_upper_80": float(np.percentile(eta_bp, 90)),
                    "etab": eta_bp, "delta": delta, "alpha": alpha,
                    "m": m,
                    "loss": loss_type, "B": B})
    df = pd.DataFrame(rows)
    dfs.append(df.copy())

In [ ]:
for i in range(len(dfs)):
    print(dfs[i])

In [ ]:
for i in range(len(dfs)):
    df = dfs[i]
    df['ratio'] = df['m'] / min(len(scaled_calcium), len(scaled_placebo))

In [ ]:
# Compute the std
for i, loss_type in enumerate(loss_types):
    df = dfs[i]
    if loss_type == 'huber':
        delta = 1.345
    elif loss_type == 'logcosh':
        delta = 1.2047
    else:
        delta = 1.4790
    theta_calcium = estimate_theta(scaled_calcium, delta=delta, loss_type=loss_type)
    theta_placebo = estimate_theta(scaled_placebo, delta=delta, loss_type=loss_type)
    var_calcium = np.sum(psi(scaled_calcium - theta_calcium, delta=delta, loss_type=loss_type)**2) / (np.sum(psi_prime(scaled_calcium - theta_calcium, delta=delta, loss_type=loss_type))**2)
    var_placebo = np.sum(psi(scaled_placebo - theta_placebo, delta=delta, loss_type=loss_type)**2) / (np.sum(psi_prime(scaled_placebo - theta_placebo, delta=delta, loss_type=loss_type))**2)
    std = np.sqrt(var_calcium + var_placebo)
    df['std'] = std

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# 1) pick your style & context
sns.set_style("whitegrid")    # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk")       # options: “paper”, “notebook”, “talk”, “poster”
palette = sns.color_palette("husl", 3)
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})

# 2) plot one loss at a time
# x_axis is ratio of m / min(n1, n2)
# y axis is eta
# use ax.fill for the confidence intervals
for i, loss_type in enumerate(loss_types):
    df = dfs[i]
    fig, ax = plt.subplots(figsize=(8, 5))
    # change stroke of marker
    sns.lineplot(data=df, x="ratio", y="eta", ax=ax, color=palette[0], markers=True, marker='x', alpha=1, zorder=6, markeredgecolor=palette[0], markersize=7, linewidth=2, linestyle='--', markeredgewidth=1.5)
    # set x ticks
    ax.set_xticks(df["ratio"])
    # ax.fill_between(df["ratio"], df["eta_lower_95"], df["eta_upper_95"], alpha=0.08, color=palette[2], label="95% CI", edgecolor=None)
    # ax.fill_between(df["ratio"], df["eta_lower_80"], df["eta_upper_80"], alpha=0.15, color=palette[2], label="80% CI", edgecolor=None)
    # use discrete error bars
    ax.vlines(df["ratio"], df['eta_lower_95'], df['eta_upper_95'], lw=6, color=palette[2], alpha=0.3, zorder=1, label="95% CI")
    ax.vlines(df["ratio"], df['eta_lower_80'], df['eta_upper_80'], lw=6, color=palette[1], alpha=0.7, label="80% CI")

    # top-edge triangle and label “∞”
    x_inf = [0.5]
    ymax = 3.2
    ax.set_ylim(-0.2, ymax)
    for xi in x_inf:
        ax.plot(xi, ymax, marker="^", ms=7, color=palette[0], linestyle="None", clip_on=False, markeredgecolor=None, zorder=10)
        ax.annotate(r"$\eta_m = \infty$", xy=(xi, ymax-0.03), xytext=(0, 7),
                    textcoords="offset points", ha="center", va="bottom",
                    color='black', fontsize=11)
    # add ticks at x = 0.5
    ax.set_xticks(list(df["ratio"]) + x_inf)
    # draw three lines for alpha = 0.05, 0.01, 0.001
    if loss_type == 'huber':
        delta = 1.345
    elif loss_type == 'logcosh':
        delta = 1.2047
    else:
        delta = 1.4790
    theta_calcium = estimate_theta(scaled_calcium, delta=delta, loss_type=loss_type)
    theta_placebo = estimate_theta(scaled_placebo, delta=delta, loss_type=loss_type)
    for alpha_level, ls in zip([0.05, 0.01, 0.001], ['--', '-.', ':']):
        # single tail
        y = norm.ppf(1 - alpha_level) * df['std'].iloc[0] - (theta_calcium - theta_placebo)
        ax.axhline(y=y, color='gray', linestyle=ls, alpha=0.7, zorder=10)
        # add text
        ax.text(-0.01, y + 0.02, r'$\alpha$' + f"={alpha_level}", color='gray', fontsize=10)
    ax.set_xlabel(r"$m / \min(x, y)$")
    ax.set_ylabel(r"$\eta_m (\hat \theta_{x} - \hat \theta_{y}, (x, y))$")
    plt.legend()
    ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.18), ncols=2, frameon=False)
    w, h = ax.figure.get_size_inches()
    ax.figure.set_size_inches(3.5, h)
    plt.savefig(f'bootstrap_eta_m_{loss_type}.pdf', bbox_inches='tight')

### Test Statistic Plot

In [ ]:
from ftbp.wald import *

In [ ]:
m = 1
# accept
two_sample_test_statistic(scaled_calcium, scaled_placebo, m, delta=1.345, loss_type='huber', bp_type='level')

In [ ]:
# Plot 1, show how the difference changes with different $m$
# Get bootstrap weights from standard exponential
np.random.seed(53)
m_grid = list(range(0, 5))
loss_types = ['huber', 'logcosh', 'concordant']

rows = []
for loss_type in loss_types:
    if loss_type == 'huber':
        delta = 1.345
    elif loss_type == 'logcosh':
        delta = 1.2047
    else:
        delta = 1.4811
    for m in m_grid:
        lowers = []
        uppers = []
        original, upper, lower = two_sample_test_statistic(scaled_calcium, scaled_placebo, m, delta=delta, loss_type=loss_type, bp_type='level')
        rows.append({'lower': lower, 'upper': upper, 'original': original, 'm': m, 'loss': loss_type, 'delta': delta})

df = pd.DataFrame(rows)

In [ ]:
df

In [ ]:
df['ratio'] = df['m'] / min(len(scaled_calcium), len(scaled_placebo))
all_df = df.copy()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# 1) pick your style & context
sns.set_style("whitegrid")    # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk")       # options: “paper”, “notebook”, “talk”, “poster”
palette = sns.color_palette("husl", 3)
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})

# 2) plot one loss at a time
# x_axis is ratio of m / min(n1, n2)
# y axis is eta
# use ax.fill for the confidence intervals
for i, loss_type in enumerate(loss_types):
    df = all_df[all_df['loss'] == loss_type]
    fig, ax = plt.subplots(figsize=(8, 5))
    # change stroke of marker
    # set x ticks
    ax.set_xticks(df["ratio"])
    # use discrete error bars
    ax.vlines(df["ratio"], df['lower'], df['upper'], lw=6, color=palette[2], alpha=0.3, zorder=2, label="[lower, upper]")
    ax.hlines(df['lower'], df['ratio'] - 0.0085, df['ratio'] + 0.0085, lw=1.6, color=palette[2], alpha=1, zorder=3)
    ax.hlines(df['upper'], df['ratio'] - 0.0085, df['ratio'] + 0.0085, lw=1.6, color=palette[2], alpha=1, zorder=3)

    # top-edge triangle and label “∞”
    x_inf = [0.5]
    ymax = 15.2
    ax.set_ylim(-0.2, ymax)
    for xi in x_inf:
        ax.plot(xi, ymax, marker="^", ms=7, color=palette[0], linestyle="None", clip_on=False, markeredgecolor=None, zorder=10)
        ax.annotate(r"$\hat \theta_{\tilde x} - \hat \theta_{\tilde y} = \infty$", xy=(xi, ymax-0.03), xytext=(0, 7),
                    textcoords="offset points", ha="center", va="bottom",
                    color='black', fontsize=11)
    # add ticks at x = 0.5
    ax.set_xticks(list(df["ratio"]) + x_inf)
    for alpha_level, ls in zip([0.05, 0.01, 0.001], ['--', '-.', ':']):
        # single tail
        y = norm.ppf(1 - alpha_level)
        ax.axhline(y=y, color='gray', linestyle=ls, alpha=0.7, zorder=10)
        # add text
        ax.text(-0.01, y + 0.02, r'$\alpha$' + f"={alpha_level}", color='gray', fontsize=10)
    ax.set_xlabel(r"$m / \min(x, y)$")
    ax.set_ylabel(r"$\frac{\hat \theta_{\tilde x} - \hat \theta_{\tilde y}}{\hat \sigma(\hat \theta_{\tilde x}, \hat \theta_{\tilde y})}$")
    plt.legend()
    ax.legend(loc='upper center', bbox_to_anchor=(0.478, -0.18), ncols=2, frameon=False)
    w, h = ax.figure.get_size_inches()
    ax.figure.set_size_inches(3.5, h)
    plt.savefig(f'test_m_{loss_type}.pdf', bbox_inches='tight')